# Network Metrics Analysis Framework

This comprehensive notebook demonstrates the capabilities of the **Network Metrics Analysis Framework** - a professional Python package for analyzing complex networks using graph theory metrics.

## 📊 What This Framework Provides

- **Comprehensive Graph Theory Metrics**: Degree distribution, clustering coefficient, path length, centrality measures, and more
- **Advanced Network Structure Analysis**: High-level analysis of computed metrics
- **Professional Visualizations**: Built-in plotting tools for comparing different network structures
- **Reproducible Research**: Complete workflow from data to publication-ready results
- **Modular Architecture**: Extensible design for adding new metrics and analysis methods

Author: Hua Cheng <trernghwhuare@aliyun.com>

---

## 🔧 Installation & Setup

Before running this notebook, ensure you have the required dependencies installed:

```bash
pip install -r requirements.txt
```

The framework supports both programmatic usage and interactive notebook exploration.

In [ ]:
import sys
import os
import glob

# Add src directory to path for local development
notebook_dir = os.getcwd()
src_path = os.path.join(notebook_dir, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import and validate dependencies
try:
    import graph_tool.all as gt
    print("✓ graph-tool imported successfully")
except ImportError as e:
    print(f"✗ Error importing graph-tool: {e}")
    print("Please ensure graph-tool is installed in your environment")

try:
    from network_metrics_package.metrics.generator import compute_and_save_metrics
    from network_metrics_package.plotting.compare_plots import (
        load_metrics, plot_violin, plot_box, plot_heatmap_corr, plot_clustermap
    )
    print("✓ network_metrics_package imported successfully")
except ImportError as e:
    print(f"✗ Error importing network_metrics_package: {e}")
    print("Please ensure the package is correctly installed or the src directory is in the path")

## 📁 Network Files Available

The framework automatically detects available network files in the current directory. Supported formats include Graph Tool (.gt) files.

In [ ]:
network_files = [
    "iT_max_plus.gt",
    "max_CTC_plus.gt", 
    "max_M2M1S1_plus.gt",
    "optimus_CTC_plus.gt",
    "optimus_M2M1S1_plus.gt"
]

existing_files = []
for f in network_files:
    if os.path.exists(f):
        existing_files.append(f)
        print(f"✓ Found network file: {f}")
    else:
        print(f"✗ Network file not found: {f}")

if not existing_files:
    print("\n⚠️  No network files found. Please ensure the .gt files are in the current directory.")

## 🎯 Advanced Usage Examples

### Example 1: Basic Network Analysis

```python
# Simple one-liner for basic analysis
from network_metrics_package.metrics.generator import NetworkMetricsGenerator

generator = NetworkMetricsGenerator()
metrics = generator.compute_all_metrics(your_network_graph)
```

### Example 2: Custom Metric Configuration

```python
# Customize which metrics to compute
metrics = generator.compute_all_metrics(
    g,
    include_centralities=True,
    include_clustering=True,
    normalize=True
)
```

### Example 3: Programmatic Visualization

```python
from network_metrics_package.plotting.compare_plots import create_comparison_plots

# Compare multiple networks
comparison_results = create_comparison_plots([metrics1, metrics2, metrics3])
```

### Example 4: Integration with Other Tools

```python
# Convert metrics to pandas DataFrame for further analysis
import pandas as pd
df = pd.DataFrame(metrics)

# Export to various formats
df.to_csv('network_metrics.csv')
df.to_json('network_metrics.json')
```

In [ ]:
def analyze_network_file(filepath, output_base_dir="metrics_analysis_outputs"):
    """
    Analyze a single network file with comprehensive error handling and logging.
    
    Parameters:
    filepath (str): Path to the network file (.gt format)
    output_base_dir (str): Base directory for all outputs
    
    Returns:
    bool: True if analysis completed successfully, False otherwise
    """
    # Extract prefix from filename
    prefix = os.path.splitext(os.path.basename(filepath))[0]
    
    # Create output directory for this network
    output_dir = os.path.join(output_base_dir, prefix)
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Analyzing {filepath}...")
    print(f"Output directory: {output_dir}")
    
    try:
        # Load the network
        print(f"  Loading network file...")
        g = gt.load_graph(filepath)
        print(f"  Network loaded: {g.num_vertices()} vertices, {g.num_edges()} edges")
        
        # Compute metrics
        print("  Computing metrics...")
        metrics, npz_path, csv_path = compute_and_save_metrics(
            g,
            out_dir=output_dir,
            prefix=prefix,
            normalize=True,
            save_files=True
        )
        print(f"  Metrics saved to {npz_path} and {csv_path}")
        
        # Load metrics for plotting
        loaded_metrics = load_metrics(npz_path)
        
        # Generate plots
        print("  Generating visualizations...")
        
        # Violin plot
        violin_file = os.path.join(output_dir, f"{prefix}_violin.png")
        plot_violin(loaded_metrics, out=violin_file)
        print(f"  Violin plot saved to {violin_file}")
        
        # Box plot
        box_file = os.path.join(output_dir, f"{prefix}_box.png")
        plot_box(loaded_metrics, out=box_file)
        print(f"  Box plot saved to {box_file}")
        
        # Correlation heatmap
        heatmap_file = os.path.join(output_dir, f"{prefix}_heatmap_corr.png")
        plot_heatmap_corr(loaded_metrics, out=heatmap_file)
        print(f"  Correlation heatmap saved to {heatmap_file}")
        
        # Clustermap (with additional error handling)
        clustermap_file = os.path.join(output_dir, f"{prefix}_clustermap.png")
        try:
            plot_clustermap(loaded_metrics, out=clustermap_file)
            print(f"  Clustermap saved to {clustermap_file}")
        except Exception as e:
            print(f"  Warning: Could not generate clustermap: {e}")
            print("  Continuing with other visualizations...")
        
        print(f"  Analysis complete for {filepath}!")
        return True
        
    except MemoryError as e:
        print(f"  Memory error analyzing {filepath}: {e}")
        print("  This may be due to large network size. Consider using a machine with more RAM.")
        return False
    except Exception as e:
        print(f"  Error analyzing {filepath}: {e}")
        import traceback
        traceback.print_exc()
        return False

## 🚀 Running the Analysis

The following code will process all available network files and generate comprehensive analysis results including:

- **Numerical metrics** in both NPZ and CSV formats
- **Violin plots** showing metric distributions
- **Box plots** for statistical summaries
- **Correlation heatmaps** revealing relationships between metrics
- **Clustermaps** for hierarchical clustering visualization

All results are organized in a structured directory hierarchy for easy access and reproducibility.

In [ ]:
if existing_files:
    # Create base output directory
    base_output_dir = "metrics_analysis_outputs"
    os.makedirs(base_output_dir, exist_ok=True)

    # Counter for successful analyses
    success_count = 0

    # Analyze each network file
    print("\nStarting analysis of your network files...")
    print("=" * 50)

    for network_file in existing_files:
        print(f"\nProcessing {network_file}...")
        if analyze_network_file(network_file, base_output_dir):
            success_count += 1

    print("\n" + "=" * 50)
    print(f"Analysis complete! {success_count} out of {len(existing_files)} networks processed successfully.")
    print(f"Results are in the '{base_output_dir}' directory, with each network in its own subdirectory.")
    
    # Show example output structure
    print("\nExample output structure:")
    print(f"{base_output_dir}/")
    print(f"  ├── iT_max_plus/")
    print(f"  │   ├── iT_max_plus_metrics.npz")
    print(f"  │   ├── iT_max_plus_metrics.csv")
    print(f"  │   ├── iT_max_plus_violin.png")
    print(f"  │   └── ...")
    print(f"  └── max_CTC_plus/")
    print(f"      ├── max_CTC_plus_metrics.npz")
      └── ...")
else:
    print("\nNo network files to process. Please add .gt files to the current directory.")

## 📈 Professional Portfolio Showcase

This framework demonstrates several key professional competencies:

### Technical Skills
- **Advanced Python Programming**: Object-oriented design, modular architecture
- **Scientific Computing**: NumPy, SciPy, Pandas integration
- **Data Visualization**: Matplotlib, Seaborn for publication-quality plots
- **Network Science**: Comprehensive graph theory implementation
- **Software Engineering**: Proper packaging, testing, documentation

### Research Capabilities
- **Reproducible Research**: Complete workflow from raw data to results
- **Academic Standards**: Proper documentation and methodology
- **Scalability**: Handles large networks with robust error handling

### Industry Applications
- **Social Network Analysis**: Community detection, influence analysis
- **Biological Networks**: Protein-protein interactions, neural connectivity
- **Infrastructure Analysis**: Transportation, communication networks
- **Financial Networks**: Risk assessment, systemic risk analysis

---

**Next Steps**: Explore the generated outputs in the `metrics_analysis_outputs` directory, or integrate this framework into your own research projects!